# 3. Calculate uplifts and save delivery

Read `../data/score/components.csv`, calculate display FMI and both uplift methods, and inspect forecast amounts.

The final cell overwrites `outputs/fmi/weighted-delivery.xlsx` and `.json` when `WRITE_DELIVERY = True`. Every brand is included, with blank scores and missing reasons where a calculation is unavailable.

In [ ]:
import math
import os
from pathlib import Path
from runpy import run_path

import pandas as pd
from dotenv import load_dotenv

from aily_project.services.fmi.configuration import input_policy
from aily_project.services.fmi.flat_delivery import flat_payload
from aily_project.services.fmi.inputs import read_inputs
from aily_project.services.fmi.models import (
    FMIResult,
    calendar_date,
    instant,
    number,
    truth,
)
from aily_project.services.fmi.policy import Policy
from aily_project.services.fmi.run_records import monthly_context
from aily_project.services.fmi.scenario import select_period, year_ago
from aily_project.services.fmi.scoring import half_up
from aily_project.services.fmi.workbooks import write_workbook

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
FMI = ROOT / "scripts/fashion-momentum-index"
load_dotenv(ROOT / ".env", override=False)
INPUTS_PATH = Path(
    os.environ.get("FMI_INPUTS_PATH", str(FMI / "inputs/FMI_inputs.json"))
)
DATA = Path(os.environ.get("FMI_DATA_DIR", str(FMI / "data")))
OUT = Path(os.environ.get("FMI_OUTPUT_DIR", str(ROOT / "outputs/fmi")))
io = run_path(str(FMI / "helpers.py"))
save_json, save_table = io["save_json"], io["save_table"]
load_table, read_snapshot = io["load_table"], io["read_snapshot"]
inputs = read_inputs(INPUTS_PATH)
policy = input_policy(INPUTS_PATH, Policy())

EXTRACT, SCORE, DELIVERY = DATA / "extract", DATA / "score", DATA / "delivery"
WRITE_DELIVERY = True

snapshot = read_snapshot(EXTRACT)
results = []
for row in load_table(SCORE / "components"):
    values = {
        k: v
        for k, v in row.items()
        if k not in {"brand_id", "analysis_id", "missing_reasons"}
    }
    reasons = row.get("missing_reasons") or ""
    results.append(
        FMIResult(
            row["brand_id"],
            row["analysis_id"],
            values,
            reasons.split("; ") if reasons else [],
        )
    )
entities = {e["entity_id"]: e for e in inputs.entities}
analyses = {a["analysis_id"]: a for a in inputs.analyses}
cutoff = instant(snapshot.metadata["extracted_at"])
print("Extraction:", cutoff)
display(
    pd.DataFrame([r.record() for r in results]).filter(
        regex="brand_id|name$|raw_fmi|fmi_status|missing_reasons"
    )
)

## 3.1 Rescale only available, selected raw headlines

Display = lower + (upper − lower) × (raw − cohort minimum) / (maximum − minimum).
The equal/single-score case uses the policy's equal score. Round half-up to one decimal.
Missing scores do not enter the cohort and stay missing.

In [ ]:
cohort_results = [
    r
    for r in results
    if r.values.get("fmi_status") == "available"
    and r.values.get("headline_selection")
    and r.values.get("raw_fmi") is not None
]
raw_values = [float(r.values["raw_fmi"]) for r in cohort_results]
lo, hi = (min(raw_values), max(raw_values)) if raw_values else (None, None)
members = sorted(str(r.analysis_id) for r in cohort_results)
cohort_id = snapshot.metadata["extracted_at"]
for result in results:
    v = result.values
    v.update(
        cohort_id=cohort_id,
        cohort_min=lo,
        cohort_max=hi,
        cohort_count=len(members),
        display_fmi=None,
        uplift_fraction=None,
        legacy_uplift_fraction=None,
        legacy_uplift_score_basis="unrounded_raw_fmi",
        uplift_score_basis="rounded_display_fmi",
    )
for result in cohort_results:
    raw_score = float(result.values["raw_fmi"])
    display_score = (
        policy.display_equal
        if lo == hi
        else (
            policy.display_lower
            + (policy.display_upper - policy.display_lower)
            * (raw_score - lo)
            / (hi - lo)
        )
    )
    result.values["display_fmi"] = half_up(display_score)
cohort = {"cohort_id": cohort_id, "members": members, "minimum": lo, "maximum": hi}

display(
    save_table(
        [
            {
                "brand_id": r.brand_id,
                "raw_fmi": r.values.get("raw_fmi"),
                "display_fmi": r.values.get("display_fmi"),
                "cohort_count": len(members),
            }
            for r in results
        ],
        DELIVERY / "rescaling",
    )
)

## 3.2 Old and new uplift rates

Old rate: zero below/at the raw-score threshold, otherwise intercept + slope × excess raw score.

New rate: maximum uplift × expm1(exponent × display / 100) / expm1(exponent).

These are business scenarios, not calibrated predictive accuracy. Cohort membership can change
the new rate even if a brand's raw score is unchanged. No forecast is needed to calculate a rate.

In [ ]:
for result in cohort_results:
    v = result.values
    raw_score, display_score = v["raw_fmi"], v["display_fmi"]
    v["legacy_uplift_fraction"] = (
        0.0
        if raw_score <= policy.legacy_uplift_threshold
        else policy.legacy_uplift_intercept
        + (raw_score - policy.legacy_uplift_threshold) * policy.legacy_uplift_per_score
    )
    v["uplift_fraction"] = (
        policy.uplift_max
        * math.expm1(policy.uplift_exponent * display_score / 100)
        / math.expm1(policy.uplift_exponent)
    )
display(
    save_table(
        [
            {
                "brand_id": r.brand_id,
                "raw_fmi": r.values.get("raw_fmi"),
                "display_fmi": r.values.get("display_fmi"),
                "legacy_uplift_fraction": r.values.get("legacy_uplift_fraction"),
                "uplift_fraction": r.values.get("uplift_fraction"),
            }
            for r in results
        ],
        DELIVERY / "uplift-rates",
    )
)

## 3.3 Select and validate the forecast baseline

The existing native-period selector retains fiscal/retail labels and flags ambiguity.
The checks below then require source-verified dossier scope, currency/units, a positive
future baseline, and compatibility with the scored financial subject.
This does not change the three periods used in notebook 2.

In [ ]:
baselines, baseline_subjects, forecast_checks = {}, {}, []
for result in results:
    v = result.values
    analysis = analyses.get(result.analysis_id)
    if not analysis or not analysis.get("forecast_target_end"):
        v.update(
            forecast_status="unavailable",
            forecast_missing_reason="missing_forecast_target",
        )
        continue
    subject = {
        **entities[result.brand_id],
        **{
            k: v[k]
            for k in (
                "light_code",
                "ticker",
                "shared_symbol",
                "financial_subject",
                "scope",
                "proxy",
                "proxy_label",
            )
            if k in v
        },
    }
    if truth(subject.get("proxy")):
        subject.update(
            proxy_authorized=bool(analysis["dossier"].get("source")),
            evidence=analysis["dossier"].get("source"),
        )
    baseline_subjects[result.brand_id] = subject
    end = calendar_date(analysis["forecast_target_end"])
    baseline, reason = select_period(
        subject,
        snapshot,
        end,
        "baseline",
        analysis["analysis_id"],
        policy.scenario_quarter_tolerance_days,
    )
    native = next(
        (
            r
            for r in snapshot.financials
            if baseline
            and r.get("source_row_id") == baseline["source_row_id"]
            and r.get("metric") == "revenue"
        ),
        {},
    )
    review = next(
        (
            r
            for r in inputs.financial_review
            if baseline
            and r["analysis_id"] == analysis["analysis_id"]
            and r["entity_id"] == result.brand_id
            and r["role"] == "baseline"
            and r.get("source_row_id") == baseline["source_row_id"]
            and truth(r.get("approved"))
            and r.get("evidence")
        ),
        {},
    )
    if baseline:
        source_scope = native.get("scope") or review.get("scope")
        if not source_scope or source_scope != v.get("scope"):
            baseline, reason = (
                None,
                "dossier_forecast_scope_not_verified_or_incompatible",
            )
        elif (
            not (native.get("currency") or review.get("currency"))
            or number(native.get("unit_multiplier") or review.get("unit_multiplier"))
            is None
        ):
            baseline, reason = None, "dossier_forecast_currency_or_units_not_verified"
    if baseline and (
        baseline["period_end"] <= cutoff.date() or baseline["amount"] <= 0
    ):
        baseline, reason = None, "target_not_future_or_nonpositive_baseline"
    if baseline and any(
        v.get(k) is not None and baseline.get(k) is not None and v[k] != baseline[k]
        for k in ("currency", "unit_multiplier")
    ):
        baseline, reason = None, "forecast_actual_currency_or_units_conflict"
    if baseline:
        baselines[result.brand_id] = baseline
        v.update(
            baseline_revenue=baseline["amount"],
            baseline_period_start=baseline["period_start"],
            baseline_period_end=baseline["period_end"],
            baseline_period_label=baseline.get("period_label"),
            baseline_fiscal_year=baseline.get("year"),
            baseline_native_quarter=baseline.get("quarter") or baseline.get("period"),
            baseline_currency=baseline.get("currency"),
            baseline_unit_multiplier=baseline.get("unit_multiplier"),
            baseline_source_row_id=baseline["source_row_id"],
            baseline_source_relation=baseline["source_relation"],
            forecast_vintage=baseline.get("available_at"),
            forecast_selection_basis="requested_end_date_native_quarter",
            forecast_calendar_alignment_status="matches_requested_end"
            if baseline["period_end"] == end
            else (
                "native_fiscal_or_retail_period_differs_from_requested_calendar_target"
            ),
            forecast_target_end_difference_days=(baseline["period_end"] - end).days,
        )
        result.financial_periods.append(baseline)
    else:
        v.update(forecast_status="unavailable", forecast_missing_reason=reason)
    forecast_checks.append(
        {
            "brand_id": result.brand_id,
            "requested_target": analysis["forecast_target_label"],
            "baseline_revenue": v.get("baseline_revenue"),
            "baseline_native_end": v.get("baseline_period_end"),
            "baseline_source_row_id": v.get("baseline_source_row_id"),
            "selection_reason": reason,
        }
    )
display(save_table(forecast_checks, DELIVERY / "baseline-selection"))

## 3.4 Same-season prior actual and adjusted amounts

Forecast YoY uses the same native season one year before the selected baseline,
not the latest actual used to score FMI. Missing prior actual leaves YoY blank.
Missing baseline leaves adjusted amounts blank without removing a valid score or uplift rate.

In [ ]:
for result in results:
    if result.brand_id not in baselines:
        continue
    v, baseline = result.values, baselines[result.brand_id]
    analysis, subject = analyses[result.analysis_id], baseline_subjects[result.brand_id]
    prior, reason = select_period(
        subject,
        snapshot,
        year_ago(baseline["period_end"]),
        "prior_year_actual",
        analysis["analysis_id"],
        policy.scenario_quarter_tolerance_days,
        native_quarter=str(baseline.get("quarter") or baseline.get("period")).upper(),
    )
    if prior:
        native = next(
            (
                r
                for r in snapshot.financials
                if r.get("source_row_id") == prior["source_row_id"]
            ),
            {},
        )
        if native.get("scope") != v.get("scope"):
            prior, reason = None, "dossier_forecast_prior_year_scope_not_verified"
    if prior and any(
        baseline.get(k) != prior.get(k)
        for k in ("currency", "unit_multiplier", "scope")
    ):
        prior, reason = None, "forecast_prior_year_currency_units_or_scope_conflict"
    if prior and prior["amount"] <= 0:
        prior, reason = None, "nonpositive_forecast_prior_year_actual"
    v.update(
        prior_year_actual_status=reason,
        prior_year_actual_period_label=prior.get("period_label") if prior else None,
        prior_year_actual_period_end=prior.get("period_end") if prior else None,
    )
    if prior:
        v["prior_year_actual"] = prior["amount"]
        v["baseline_forecast_yoy"] = baseline["amount"] / prior["amount"] - 1
        result.financial_periods.append(prior)
    if v.get("uplift_fraction") is None:
        v.update(
            forecast_status="unavailable",
            forecast_missing_reason="missing_available_display_fmi",
        )
        continue
    amount = baseline["amount"]
    v["adjustment"] = amount * v["uplift_fraction"]
    v["adjusted_forecast"] = amount + v["adjustment"]
    v["legacy_adjustment"] = amount * v["legacy_uplift_fraction"]
    v["legacy_adjusted_forecast"] = amount + v["legacy_adjustment"]
    v["forecast_status"] = "available"
    if prior:
        v["adjusted_forecast_yoy"] = v["adjusted_forecast"] / prior["amount"] - 1
        v["legacy_adjusted_forecast_yoy"] = (
            v["legacy_adjusted_forecast"] / prior["amount"] - 1
        )
for result in results:
    v = result.values
    v["narrative"] = (
        (
            f"Raw FMI {v['raw_fmi']:.1f} ({v['raw_classification']}); "
            f"display {v['display_fmi']:.1f}; "
            f"scenario uplift {100 * v['uplift_fraction']:.2f}%."
        )
        if v.get("display_fmi") is not None
        else "FMI unavailable: " + "; ".join(dict.fromkeys(result.gaps))
    )
display(
    save_table(
        [r.record() for r in results], DELIVERY / "forecast-calculations"
    ).filter(
        regex="brand_id|baseline_revenue|adjustment|adjusted_forecast|prior_year_actual$|yoy|forecast_status|forecast_missing_reason"
    )
)

## 3.5 Prepare delivery

Keep every brand, including available components and missing reasons. The flat-table helper adds app fields and observed context; it does not change scores.

In [ ]:
contexts = monthly_context(inputs, snapshot)
save_table(contexts, DELIVERY / "monthly-social-context")
flat = flat_payload(results, inputs, snapshot, contexts)
all_rows = flat["sheets"][0]["rows"]
display(
    save_table(all_rows, DELIVERY / "weighted-delivery").filter(
        regex="brand_id|name$|ir_score|social_score|trend_score|raw_fmi|display_fmi|fmi_status|forecast_status|missing_reasons"
    )
)

## 3.6 Save Excel and JSON

With `WRITE_DELIVERY = True`, both files are replaced by this run, even if all headline scores are blank. The dossier delivery is not changed by these notebooks.

In [ ]:
summary = {
    "brands": len(all_rows),
    "available_fmi": sum(r.get("fmi_status") == "available" for r in all_rows),
    "adjusted_forecasts": sum(
        r.get("forecast_status") == "available" for r in all_rows
    ),
}
display(pd.DataFrame([summary]))
if WRITE_DELIVERY:
    OUT.mkdir(parents=True, exist_ok=True)
    write_workbook(
        flat, OUT / "weighted-delivery.xlsx", previews=True, work_dir=OUT / ".work"
    )
    save_json(all_rows, OUT / "weighted-delivery.json")
    print("Replaced Excel and JSON in", OUT)
else:
    print("CSV tables saved. Delivery files were not changed.")